# 02 — E1: power and strata (narrowed scope)

No API calls. See `src/stats_checks.py` docstring / `BLOCKED_ANALYSES` for why two of the three planned E1 pieces cannot run from aggregated-only data:

- Paired McNemar (Flash S8 vs Claude S8) → substituted with an **unpaired** two-proportion z-test (weaker evidence, explicitly labeled).
- Category stratification on MT-Bench → **blocked**, no per-instance category field in the aggregated cache.

What this notebook *can* do from aggregated data alone: check the paper's own "N=400 detects ~4-5 points" power claim (W2) against a first-principles Wald CI, and check CI overlap for Flash S8 vs Claude S8 (W7).

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
UPSTREAM = PROJECT_ROOT / "data" / "upstream"
TABLE1 = UPSTREAM / "analysis" / "figures" / "table1_main_results.csv"

from src.stats_checks import BLOCKED_ANALYSES, check_mde_claim, ci_overlap, two_proportion_ztest, wald_ci_half_width

print("Blocked analyses in this notebook:")
for k, v in BLOCKED_ANALYSES.items():
    print(f"- {k}: {v}")

In [ ]:
# W2: is the paper's "N=400 detects ~4-5 points" claim consistent with a
# first-principles Wald CI half-width, and with the bootstrap CIs it published?
print(f"Theoretical Wald half-width at n=400, p=0.6: {wald_ci_half_width(400, 0.6):.4f} ({wald_ci_half_width(400, 0.6)*100:.1f} points)")

mde_report = check_mde_claim(TABLE1, n=400, tolerance=0.015)
assert mde_report["within_tolerance"].all()
print("PASS: all 25 published CI half-widths match the n=400 Wald formula within 1.5 points")
mde_report

In [ ]:
# W7: Flash S8 vs Claude S8 on MT-Bench -- CI overlap (strong evidence) +
# unpaired proportion test (weak substitute for the McNemar the plan wanted).
df = pd.read_csv(TABLE1).set_index("Model")
overlap = ci_overlap(df.loc["Gemini 2.5 Flash", "Combined Budget"], df.loc["Claude Sonnet 4", "Combined Budget"])
print("CI overlap check:", overlap)

ztest = two_proportion_ztest(n1=400, p1=0.710, n2=400, p2=0.695)
print("Unpaired two-proportion z-test (NOT McNemar, see BLOCKED_ANALYSES):", ztest)

assert overlap["overlaps"] is True
assert ztest["significant_at_0.05"] is False
print("\nConclusion: no statistical basis found for 'Flash S8 beats Claude S8' -- confirms W7's suspicion.")

## W-item status after this notebook

| ID | Trạng thái sau E1 (thu hẹp) |
| --- | --- |
| W2 ("không có tác dụng" có thể do thiếu power) | **Đã xác nhận bằng công thức**: Wald half-width lý thuyết ở n=400 (≈4,5-5 điểm) khớp với toàn bộ 25 CI đã công bố trong Table 1 (sai số ≤1,5 điểm). Xác nhận claim gốc của tác giả là có cơ sở toán học, không chỉ là suy đoán |
| W7 (Flash S8 vs Claude S8 không có cơ sở thống kê) | **Đã xác nhận**: hai CI chồng lấn 0,075; z-test hai tỷ lệ (không ghép cặp, yếu hơn McNemar) cho p > 0,05 → không bác bỏ được giả thuyết "không khác nhau". Kết luận "model tầm trung thắng frontier" trong bài chưa có cơ sở thống kê vững |
| E1 phân tầng theo category MT-Bench | **Bị chặn hoàn toàn** — không có trường category trong aggregated cache |

## Cổng 1 — có nên đi tiếp sang E2–E5 (tốn API) không?

Theo tiêu chí ở mục 8 của master plan: E0 không sụp (Table 1 khớp 100%), nhưng E0 đầy đủ và một phần E1 (McNemar ghép cặp, phân tầng category) không chạy được vì thiếu `results/raw/`. W7 đã có bằng chứng đủ mạnh (CI chồng lấn) để nghi ngờ claim "model tầm trung thắng frontier" của bài — đây là input quan trọng cho Cổng 3 (approach). Quyết định đi tiếp E2 (position bias, chi phí thấp) nên do bạn chốt, vì nó bắt đầu tốn API thật.